# Phase 0: Identify Surgery and Radiation Therapy Prevalence

## Purpose
This notebook analyzes the prevalence of surgery and radiation therapy in Head and Neck Cancer (HNC) patients by:
- Identifying surgical procedures from procedure codes
- Detecting radiation therapy from procedure codes and clinical notes
- Calculating prevalence rates across the cohort
- Analyzing treatment modality combinations (surgery only, radiation only, both, neither)

## Inputs
- Raw PCDM tables:
  - `PCDM_PROCEDURES` - Procedure records with CPT/ICD codes
  - `PCDM_DIAGNOSIS` - Diagnosis records to identify HNC patients
  - `PCDM_DEMOGRAPHIC` - Patient demographics
- Translation dictionaries from `input_data/ICD_RX translation/`
- (Optional) Clinical notes if available

## Outputs
- **Visualizations**: Bar charts showing treatment modality distributions
- **Summary tables**: Counts and percentages for each treatment type
- **Cohort characteristics**: Demographics by treatment modality

## Key Treatment Modalities Identified
- **Surgery**: Surgical resection procedures (e.g., laryngectomy, glossectomy)
- **Radiation therapy**: External beam radiation, brachytherapy
- **Combined modality**: Surgery + radiation
- **Other treatments**: Chemotherapy alone, observation

## What This Notebook Does
1. Extracts all procedure codes from HNC patients
2. Filters for procedures related to surgery and radiation
3. Classifies patients by treatment modality received
4. Calculates prevalence rates for each modality
5. Analyzes timing of treatments relative to diagnosis
6. Creates visualizations showing treatment patterns
7. Compares patient characteristics across treatment groups

## Key Variables Created
- `surgery_indicator` - Binary flag for surgical procedures
- `radiation_indicator` - Binary flag for radiation therapy
- `treatment_modality` - Categorical variable (surgery/radiation/both/neither)

## When to Run
- **Exploratory analysis phase** to understand standard of care patterns
- To validate that expected treatments appear in the data
- Before outcome analysis to understand potential confounders
- To create treatment modality stratification variables

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
import math

# Set pandas view options to display all rows and columns
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


In [ ]:
PCDM_PROCEDURES = pd.read_csv('/path/to/IRB_DATA_DRIVE/PCDM_2023_03_09/IRB_PCDM_PROCEDURES.csv')
print("15/17 Read in PCDM_PROCEDURES")

In [ ]:
PCDM_PROCEDURES

### Read in procedure translation

In [ ]:
### Translation dictionary for procedures
ProcedCodeTranslationDf = pd.read_csv('ICD_RX translation/HNC procedure ICD Translation DF.csv')
procedTranlateDict=dict(zip(ProcedCodeTranslationDf['ICD_CODES'], ProcedCodeTranslationDf['DESCRIPTION']))

In [ ]:
# Group by dx_code and count unique patient_id
result = PCDM_PROCEDURES.groupby('PX')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['PX', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result['TRANSLATED_DESCRIPTION'] = result['PX'].map(procedTranlateDict, na_action='ignore')

In [ ]:
procedTranlateDict['88305']

In [ ]:
result[0:25]

In [ ]:
### Surgery filtering
surgeryICD = []
searchingCriteria = ['surger', 'ectomy', 'opsy', 'pexy', 'tomy', 'desis']
for val in tqdm(procedTranlateDict.keys()):
    for criteria in searchingCriteria:
        if str.lower(criteria) in str.lower(str(procedTranlateDict[val])):
            surgeryICD.append(val)

In [ ]:
### radiation procedure filtering
radiationICD = []
searchingCriteria = ['radio', 'radiats', 'radia']
for val in tqdm(procedTranlateDict.keys()):
    for criteria in searchingCriteria:
        if str.lower(criteria) in str.lower(str(procedTranlateDict[val])):
            radiationICD.append(val)

In [ ]:
filterICDs = surgeryICD + radiationICD

In [ ]:
radiationICD

In [ ]:
len(set(PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(filterICDs)]['PATID']))

In [ ]:
procedCodeTranslationDF = pd.DataFrame.from_dict(procedTranlateDict, orient = 'index',columns=['DESCRIPTION']).reset_index(names='ICD_CODES')
procedCodeTranslationDF

In [ ]:
procedCodeTranslationDF[procedCodeTranslationDF['ICD_CODES'].isin(surgeryICD)]

In [ ]:
procedCodeTranslationDF[procedCodeTranslationDF['ICD_CODES'].isin(radiationICD)]

In [ ]:
temp_proced = PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(surgeryICD)]
# Group by dx_code and count unique patient_id
result = temp_proced.groupby('PX')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['PX', 'unique_patient_count']
result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
result

In [ ]:
### after review of codes and descriptions, we will be using the following codes as our surgery codes for the rest of the analysis. 
# These are the top most prevalent surgery codes in our dataset. 
# We will be using these codes to identify patients who have undergone surgery and to identify the type of surgery they have undergone.
### this is a more focused list from above with a few additions of codes that were found through further analysis of the data and review of the descriptions.

# These procedure codes were included to broadly capture surgical and procedure-related interventions
# that may be associated with medication use and thus act as potential confounders in downstream analyses.
# The list intentionally spans multiple categories—including definitive surgical procedures (e.g., resections,
# neck dissections, airway surgeries), perioperative or supportive interventions (e.g., tracheostomy, feeding
# tube placement), pathology/laboratory processing (e.g., 883xx series), and ancillary HCPCS/supply or drug
# administration codes—to ensure comprehensive coverage of the clinical workflow surrounding treatment.
# While not all codes directly represent therapeutic interventions, each may reflect a clinical state,
# complication risk, or care pathway (e.g., infection risk, pain burden, nutritional support, airway management,
# or postoperative monitoring) that influences medication utilization and patient outcomes. Therefore, inclusion
# of these heterogeneous procedure groups allows for more complete identification and adjustment of confounding
# factors, improving the interpretability and robustness of associations between treatments and outcomes.
top_surgery_codes = ['88305',
 '99024',
 '88307',
 '88331',
 '38724',
 '88309',
 '88311',
 '49440',
 '31600',
 '88304',
 '43653',
 '43246',
 '41120',
 '49450',
 '88332',
 '43762',
 '88300',
 '31237',
 'S2900',
 '31502',
 '17000',
 '41150',
 '43.11',
 '17311',
 '17003',
 '42826',
 '35701',
 '17110',
 '31.1',
 '31360',
 '42890',
 '31225',
 '41130',
 '17312',
 '49465',
 '97.02',
 '31615',
 '25.2',
 '41153',
 '69433',
 '88333',
 '00142',
 '61590',
 '38700',
 '47562',
 '43830',
 '61581',
 '88302',
 '32666',
 '21070',
 '76.31',
 '31267',
 '41155',
 '45.42',
 'L8501',
 '31040',
 '93286',
 '88334',
 '32674',
 '32551',
 '43030',
 '49650',
 '43.19',
 '17004',
 '69436',
 '21470',
 '17262',
 '49452',
 '29.33',
 '19301',
 '41135',
 '49446',
 '31276',
 '15839',
 '32663',
 '93287',
 '23.19',
 '31288',
 '28.2',
 '38720',
 '76098',
 '67414',
 '33508',
 '97.23',
 '43242',
 '15004',
 '31255',
 '17313',
 '61796',
 '60240',
 '21198',
 '60252',
 '41140',
 '13160',
 '60220',
 '29826',
 '42960',
 '17261',
 '31367',
 '21047',
 '43274',
 '61510',
 '66982',
 '30150',
 '31603',
 '21685',
 '26.31',
 '92941',
 '26.32',
 '69420',
 '52234',
 '29827',
 '22614',
 '17272',
 '69990',
 '22551',
 '92626',
 '20930',
 'G0416',
 '61583',
 '43262',
 '66821',
 '63047',
 '17263',
 '22.63',
 '43259',
 '43870',
 '35301',
 '63030',
 '49451',
 '42962',
 '31259',
 '51703',
 '22853',
 '31610',
 '77372',
 '44180',
 '31238',
 '22.64',
 'J9280',
 '37225',
 '19303',
 '30.4',
 '29823',
 '29881',
 'G0269',
 '17282',
 '32668',
 '99195',
 '44186',
 '15002',
 '30.3',
 '60500',
 '31395',
 '38745',
 '32667',
 '52235',
 '44160',
 '61591',
 '52601',
 '44205',
 '31365',
 '21046',
 '44213',
 '44970',
 '29824',
 '63048',
 '51595',
 '22552',
 '31254',
 '37799',
 'G8918',
 '31825',
 'C1714',
 '61580',
 '29.31',
 '55866',
 '58571',
 '41830',
 '22600',
 '52224',
 '44015',
 '68720',
 '20985',
 '35371',
 '29880',
 'L3260',
 '97.03',
 '61645',
 '15003',
 '31605',
 '20936',
 '52240',
 '29828',
 '17271',
 '61797',
 '01402',
 '50432',
 '00103',
 '49441',
 'C1724',
 '00.33',
 'A4550',
 '31256',
 '43116',
 '47600',
 '67412',
 '17314',
 '39.61',
 '67036',
 '67108',
 '67400',
 '55840',
 '51.23',
 '17273',
 '69502',
 '44139',
 '44120',
 '83.45',
 '38570',
 '32652',
 '32650',
 '31253',
 '31820',
 '00.31',
 '52281',
 '06.4',
 '58558',
 '58661',
 '22.2',
 '20931',
 '17281',
 '50205',
 '92933',
 '48.36',
 '44207',
 '25.4',
 'A4649',
 '01630',
 '44145',
 'C1757',
 'J7315',
 '43763',
 '00176',
 '26.30',
 '51.85',
 '51045',
 '52648',
 '69930',
 '22.41',
 '19307',
 '15005',
 '97.51',
 '50360',
 '30.29',
 '49651',
 '46221',
 '30580',
 '31.74',
 '42140',
 '44300',
 '44310',
 '44625',
 '45110',
 '23430',
 '46.39',
 '32480',
 '47370',
 '49321',
 '47563',
 '49000',
 '40500',
 '43281',
 '43644',
 '32651',
 '32653',
 'J0878',
 '44140',
 '42821',
 '32669',
 '44204',
 '38746',
 '92937',
 '61518',
 '56620',
 '86612',
 '38571',
 '69421',
 '58573',
 '46.32',
 '63620',
 '63045',
 '36832',
 '67042',
 '43276',
 '19340',
 '17315',
 '31390',
 '01400',
 '28285',
 '30160',
 '22612',
 '31830',
 '23120',
 '32.41',
 '22610',
 '17283',
 '00.39',
 '92.32',
 '88329',
 '50435',
 '50543',
 '57283',
 '43107',
 '22633',
 '42961',
 '92.30',
 '29876',
 '31287',
 '06.2',
 '92943',
 '63277',
 '42.41',
 '42330',
 '50389',
 '21085',
 '85.23',
 '66852',
 '27310',
 '27132',
 '27602',
 '17276',
 '69530',
 '46922',
 '46924',
 '28008',
 '44188',
 '47143',
 '69643',
 '17264',
 '44143',
 '48150',
 '44050',
 '17260',
 '49324',
 '49460',
 '58262',
 '63015',
 '45990',
 '60210',
 '60260',
 'G8916',
 '36831',
 '60650',
 '60.5',
 '61585',
 '00540',
 '20937',
 '59400',
 '61798',
 '37187',
 '37227',
 '61312',
 '38.82',
 '35372',
 '32.49',
 '21048',
 '21365',
 '22.52',
 '25.3',
 '69644',
 '43775',
 '77.81',
 '44150',
 '67335',
 '50693',
 '17111',
 '28750',
 '43832',
 '49002',
 '76.64',
 '44.38',
 '25020',
 '76.39',
 '17.35',
 '19302',
 '67041',
 '50546',
 '43253',
 '35875',
 '01.25',
 '17.56',
 '44141',
 '69635',
 'C9602',
 '19342',
 '61618',
 '34201',
 '60271',
 '28122',
 '67311',
 '21620',
 '22854',
 '31370',
 '44372',
 '54161',
 '17270',
 '31230',
 '39.27',
 '19371',
 '46255',
 '31240',
 '32120',
 '67825',
 '61582',
 '63035',
 '00126',
 '21280',
 '20.49',
 '38.18',
 '54065',
 '21050',
 '21049',
 '36833',
 '22585',
 '32110',
 '52276',
 '51705',
 '61500',
 '41820',
 '44320',
 '14.74',
 '67450',
 '54056',
 '44202',
 '10040',
 '47.01',
 '46.20',
 '47120',
 '26037',
 '17284',
 '17286',
 '46270',
 '67445',
 '46260',
 '61320',
 '21282',
 '58260',
 '67040',
 '20.01',
 '63267',
 '54.11',
 '63265',
 '54060',
 '62121',
 '63081',
 '55.51',
 '63046',
 '22554',
 '56.51',
 '56501',
 '56515',
 '22206',
 '56630',
 '57.17',
 '57.71',
 '58662',
 '62100',
 '19316',
 '65855',
 '49659',
 '50230',
 '61697',
 '50545',
 '50548',
 '50688',
 '50695',
 '65820',
 '19370',
 '51.22',
 '65400',
 '22630',
 '60.29',
 '64774',
 '69220',
 '22595',
 '52.7',
 '51710',
 '45397',
 '86.24',
 '38.83',
 '43.99',
 '29822',
 '43112',
 '43117',
 '31300',
 '28725',
 '85.6',
 '16.09',
 '37765',
 '37229',
 'A7526',
 '37184',
 '43332',
 '85.0',
 '43420',
 'A4369',
 '38120',
 '31375',
 '31070',
 '0275T',
 '06.39',
 '31239',
 '09.81',
 '31087',
 '31084',
 '41145',
 '31051',
 '15850',
 '31241',
 '42335',
 '15829',
 '97.62',
 '38564',
 '42825',
 '31.29',
 '36905',
 '31200',
 '27707',
 '32655',
 '17266',
 '33999',
 '73530',
 '44346',
 '32662',
 '32659',
 'C9607',
 '44227',
 '27870',
 '44121',
 '28124',
 '76.41',
 '17280',
 '44210',
 '00865',
 '45.73',
 'C9604',
 '77.88',
 '45.76',
 '45190',
 '31382',
 'G8911',
 '28288',
 '69636',
 '97.41',
 '00542',
 '01951',
 '21.69',
 '0238T',
 '97.04',
 '20.42',
 '00162',
 '00.35',
 '96.36',
 '63042',
 '63003',
 '0184T',
 '63020',
 '01638',
 '01740',
 '61619',
 'C9605',
 '93750',
 '01756',
 'A4625',
 'C9606',
 '62201',
 '01830',
 '62350',
 '61597',
 '61592',
 '63001',
 'C9766',
 '01274',
 '61698',
 '01842',
 '61586',
 '20.92',
 '63741',
 '63082',
 '67314',
 '17.55',
 '67010',
 '67015',
 '83.13',
 '83.02',
 '80.05',
 '19305',
 '77.89',
 '67110',
 '67113',
 '76.65',
 '67312',
 '67318',
 '63102',
 '67334',
 '19020',
 '69633',
 '69631',
 '67420',
 '69604',
 '21194',
 '69511',
 '67715',
 '17274',
 '68.41',
 '68550',
 '85.34',
 '66761',
 '66635',
 '85.41',
 '07.22',
 '92627',
 '63275',
 '63276',
 '63300',
 '15828',
 '63303',
 '92.31',
 '68760',
 '15830',
 '64738',
 '15832',
 '65.41',
 '15877',
 '15878',
 '65771',
 '16035',
 '16036',
 '17.32',
 '88249',
 '17.36',
 '66.62',
 '85.44',
 '66170',
 '85.43',
 '69601',
 '32505',
 '61530',
 '43130',
 '43282',
 '28289',
 '43280',
 '43277',
 '00.32',
 '28291',
 '28300',
 '28306',
 '43180',
 '29806',
 '43335',
 '43.3',
 '29825',
 '29848',
 '42970',
 '29874',
 '42830',
 '31030',
 '42820',
 '42340',
 '43287',
 '43352',
 '61514',
 '44020',
 '28.7',
 '44187',
 '28010',
 '28060',
 '28111',
 '44158',
 '28112',
 '28118',
 '44021',
 '28140',
 '43620',
 '44.32',
 '28150',
 '28230',
 '43820',
 '43774',
 '28232',
 '28270',
 '43633',
 '43632',
 '31050',
 '42.52',
 '42.42',
 '31420',
 '37.11',
 '36906',
 '35876',
 '35351',
 '35303',
 '34111',
 '34101',
 '34.03',
 '33572',
 '33366',
 '42.11',
 '33202',
 '33030',
 '32671',
 '32036',
 '32097',
 '32320',
 '32442',
 '32445',
 '32560',
 '37185',
 '37186',
 '31368',
 '37766',
 '31080',
 '31081',
 '31085',
 '31090',
 '41010',
 '41.5',
 '40819',
 '39010',
 '38770',
 '38760',
 '38747',
 '38740',
 '31257',
 '38572',
 '31292',
 '31295',
 '38562',
 '38100',
 '38.85',
 '27709',
 '27610',
 '44322',
 '54530',
 '57.18',
 '22208',
 '55873',
 '22212',
 '55845',
 '22216',
 '55250',
 '55.4',
 '54860',
 '54057',
 '51550',
 '53215',
 '52649',
 '22222',
 '52630',
 '22556',
 '22558',
 '22590',
 '52214',
 '32486',
 '57065',
 '58146',
 '58150',
 '58552',
 '61501',
 '61343',
 '61304',
 '60540',
 '60521',
 '60520',
 '60502',
 '60254',
 '21433',
 '60225',
 '21750',
 '60212',
 '59409',
 '59151',
 '59.94',
 '58720',
 '22.31',
 '22.42',
 '58563',
 '51590',
 '51040',
 '44373',
 '46060',
 '47490',
 '26110',
 '47125',
 '46946',
 '46945',
 '46916',
 '46280',
 '46258',
 '26850',
 '26860',
 '22634',
 '46.21',
 '46.10',
 '45541',
 '45111',
 '27524',
 '44800',
 '44626',
 '27605',
 '44604',
 '26040',
 '25230',
 '47605',
 '25040',
 '50800',
 '50728',
 '50542',
 '23.73',
 '50081',
 '50080',
 '23130',
 '23180',
 '23405',
 '23485',
 '23600',
 '23605',
 '23615',
 '24147',
 '49325',
 '49322',
 '25.91',
 '49255',
 '48.76',
 'V2790',
 '41120']

In [ ]:
temp_proced = PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(radiationICD)]
# Group by dx_code and count unique patient_id
result = temp_proced.groupby('PX')['PATID'].nunique().reset_index()
# Rename columns for clarity
result.columns = ['PX', 'unique_patient_count']
result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
result

In [ ]:
### After review of codes and descriptions, we will be using the following codes as our radiation therapy codes for the rest of the analysis.
### they were furhter limited as many radiation codes focus on viewing rather than treamtnet as we are interested in
top_radiation_codes = ['77427', '77336', '77263', '77301', '77338', '77300', '77386', '77470', '77290', '77280', '77412']

In [ ]:
temp_proced = PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(filterICDs)]
# Group by dx_code and count unique patient_id
result = temp_proced.groupby('PX')['PATID'].nunique().reset_index()
# Rename columns for clarity
result.columns = ['PX', 'unique_patient_count']
result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
result

In [ ]:
result['TRANSLATED_DESCRIPTION'] = result['PX'].map(procedTranlateDict, na_action='ignore')
result[0:50]

In [ ]:
list(result[0:50]['PX'])

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['PX'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()

In [ ]:
temp_proced = PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(radiationICD)]
# Group by dx_code and count unique patient_id
result = temp_proced.groupby('PX')['PATID'].nunique().reset_index()
# Rename columns for clarity
result.columns = ['PX', 'unique_patient_count']
result.sort_values(by='unique_patient_count', ascending=False, inplace=True)

# Plot the radiation data
plot_data = result[result['PX'].isin(top_radiation_codes)].head(25)
plt.bar(plot_data['PX'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Radiation Therapy Code')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
temp_proced = PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(surgeryICD)]
# Group by dx_code and count unique patient_id
result = temp_proced.groupby('PX')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['PX', 'unique_patient_count']
result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# Plot the data
plot_data = result[result['PX'].isin(top_surgery_codes)].head(25)
plt.bar(plot_data['PX'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Surgery Code')
plt.show()